In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/student_performance_dataset.csv")
print("First 5 rows of the dataset:")
print(df.head())
print("\nStatistical Summary:")
print(df.describe())

First 5 rows of the dataset:
  Student_ID  Gender  Study_Hours_per_Week  Attendance_Rate  Past_Exam_Scores  \
0       S147    Male                    31        68.267841                86   
1       S136    Male                    16        78.222927                73   
2       S209  Female                    21        87.525096                74   
3       S458  Female                    27        92.076483                99   
4       S078  Female                    37        98.655517                63   

  Parental_Education_Level Internet_Access_at_Home Extracurricular_Activities  \
0              High School                     Yes                        Yes   
1                      PhD                      No                         No   
2                      PhD                     Yes                         No   
3                Bachelors                      No                         No   
4                  Masters                      No                        Yes  

In [ ]:
numeric_cols = ['Study_Hours_per_week', 'Attendance_Rate','Past_Exam_Scores', 'Final_Exam_Scores']

In [ ]:
df.loc[0, 'Study_Hours_per_week'] = 120
df.loc[0, 'Attendance_Rate'] = 250
df.loc[0, 'Past_Exam_Scores'] = -20
df.loc[0, 'Final_Exam_Scores'] = 200
print("\nInserted a few artificial outlier values for demonstration")


Inserted a few artificial outlier values for demonstration


In [ ]:
print("\n-----IQR Method -----")
for col in numeric_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 -Q1
  lower = Q1 - 1.5 * IQR
  upper = Q3 + 1.5 *IQR

  outliers = df[(df[col] < lower) | (df[col] > upper)]



-----IQR Method -----


In [ ]:
print("\n----- IQR Method -----")

for col in numeric_cols:
    # Exclude missing values
    data = df[col].dropna()

    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Standard IQR outlier detection
    outlier_mask = (data < lower) | (data > upper)
    outliers = data[outlier_mask]

    print(f"\nColumn: {col}")
    print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"Limits: {lower:.2f} to {upper:.2f}")
    print(f"Missing values: {df[col].isna().sum()}")
    print(f"Outliers found: {len(outliers)}")
    print("Outlier values:", outliers.tolist())


----- IQR Method -----

Column: Study_Hours_per_week
Q1: 120.00, Q3: 120.00, IQR: 0.00
Limits: 120.00 to 120.00
Missing values: 707
Outliers found: 0
Outlier values: []

Column: Attendance_Rate
Q1: 67.55, Q3: 89.50, IQR: 21.95
Limits: 34.62 to 122.44
Missing values: 0
Outliers found: 1
Outlier values: [250.0]

Column: Past_Exam_Scores
Q1: 65.00, Q3: 91.00, IQR: 26.00
Limits: 26.00 to 130.00
Missing values: 0
Outliers found: 1
Outlier values: [-20]

Column: Final_Exam_Scores
Q1: 200.00, Q3: 200.00, IQR: 0.00
Limits: 200.00 to 200.00
Missing values: 707
Outliers found: 0
Outlier values: []


In [ ]:
print("\n ---- Z-Score Method ------")
for col in numeric_cols:
  z_scores = (df[col] - df[col].mean()) / df[col].std()
  outliers = df[z_scores.abs() > 3]
  print(f"{col}:{len(outliers)} outlier(s) found (|z| > 3)")


 ---- Z-Score Method ------
Study_Hours_per_week:0 outlier(s) found (|z| > 3)
Attendance_Rate:1 outlier(s) found (|z| > 3)
Past_Exam_Scores:1 outlier(s) found (|z| > 3)
Final_Exam_Scores:0 outlier(s) found (|z| > 3)


In [ ]:
df_treated = df.copy()
print("\n------Treating Outliers (Capping using IQR limits)------")
for col in numeric_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 -Q1
  lower = Q1 - 1.5 * IQR
  upper = Q3 + 1.5 *IQR
  df_treated[col] = df_treated[col].clip(lower, upper)
  print(f"{col}: Outliers capped between {lower:.2f} and {upper:.2f}")


------Treating Outliers (Capping using IQR limits)------
Study_Hours_per_week: Outliers capped between 120.00 and 120.00
Attendance_Rate: Outliers capped between 34.62 and 122.44
Past_Exam_Scores: Outliers capped between 26.00 and 130.00
Final_Exam_Scores: Outliers capped between 200.00 and 200.00


In [ ]:
print("\nStatistics BEFORE treatment:")
print(df[numeric_cols].describe())

print("\nStatistics AFTER treatment:")
print(df_treated[numeric_cols].describe())



Statistics BEFORE treatment:
       Study_Hours_per_week  Attendance_Rate  Past_Exam_Scores  \
count                   1.0       708.000000        708.000000   
mean                  120.0        78.364406         77.721751   
std                     NaN        15.235039         14.861748   
min                   120.0        50.116970        -20.000000   
25%                   120.0        67.550094         65.000000   
50%                   120.0        79.372680         79.000000   
75%                   120.0        89.504232         91.000000   
max                   120.0       250.000000        100.000000   

       Final_Exam_Scores  
count                1.0  
mean               200.0  
std                  NaN  
min                200.0  
25%                200.0  
50%                200.0  
75%                200.0  
max                200.0  

Statistics AFTER treatment:
       Study_Hours_per_week  Attendance_Rate  Past_Exam_Scores  \
count                   1.0       708

In [ ]:
df_treated.to_csv("student_performance_dataset_treated.csv", index=False)
print("\nTreated dataset saved as 'student_performance_dataset_treated.csv' ")
print("Program executed successfully.")


Treated dataset saved as 'student_performance_dataset_treated.csv' 
Program executed successfully.
